# Image Classification Using Convolutional Neural Network (CNN) with CIFAR-10

**A Beginner-Friendly Deep Learning Laboratory Notebook (PyTorch + torchvision)**

---

## About this dataset

This notebook uses the **official CIFAR-10 "Python version" dataset files** — the same raw pickle files distributed by the University of Toronto (`data_batch_1` … `data_batch_5`, `test_batch`, `batches.meta`) — loaded **directly from local files you upload to Colab**, instead of letting `torchvision` auto-download them. This is useful when you already have the dataset saved locally (e.g. `cifar-10-batches-py/`) and want to reuse those exact files rather than re-downloading.

**Files you need to upload to Colab (see "Files to Upload" note in Section 2):**

| File | Needed? |
|---|---|
| `data_batch_1` | ✅ Yes |
| `data_batch_2` | ✅ Yes |
| `data_batch_3` | ✅ Yes |
| `data_batch_4` | ✅ Yes |
| `data_batch_5` | ✅ Yes |
| `test_batch` | ✅ Yes |
| `batches.meta` | ✅ Yes (contains the class names) |
| `readme.html` | ❌ Not needed (just a redirect page, safe to skip) |

So: **7 files total**, all without any file extension. Section 2 below writes a small custom PyTorch `Dataset` class that reads these pickle files directly (the same format used internally by `torchvision.datasets.CIFAR10`, just loaded by hand here).

## Learning Objectives

By completing this lab, you should understand:

1. What an image classification problem is
2. What a dataset is
3. What CIFAR-10 is
4. What training, validation, and test data mean
5. What an image tensor is
6. What batch size means
7. What an epoch means
8. What a neural network does
9. What a CNN (Convolutional Neural Network) is
10. Convolution layer
11. Pooling layer
12. ReLU activation
13. Fully Connected (Linear) layer
14. Forward propagation
15. Loss function
16. Backpropagation
17. Gradient descent
18. Optimizer
19. Learning rate
20. Training a CNN
21. Validation
22. Testing
23. Accuracy
24. Confusion matrix
25. Overfitting
26. Training loss vs. validation loss
27. How to make predictions on new images

**How to use this notebook:** Run the cells from top to bottom in Google Colab (`Runtime → Run all`, or run cell-by-cell with `Shift+Enter`). Every code cell is preceded by a short explanation. Read the explanation, then look at the code, then run it.


## Section 1 — Import Libraries

We start by importing everything we need:

- **torch** — the core PyTorch library (tensors, autograd, neural network building blocks).
- **torchvision** — gives us ready-made datasets (like CIFAR-10) and image transforms.
- **torch.nn** — contains layers (Conv2d, Linear, etc.) used to build neural networks.
- **torch.optim** — contains optimizers (Adam, SGD, etc.) that update the network's weights.
- **matplotlib** — for plotting images and graphs.
- **numpy** — for numerical array operations.
- **sklearn.metrics** — for the confusion matrix and classification report.
- **seaborn** — to draw a nice-looking heatmap of the confusion matrix.

### Why GPU matters for Deep Learning
Training a CNN involves millions of multiply-and-add operations (matrix multiplications) repeated over thousands of batches. A **GPU (Graphics Processing Unit)** can perform many of these operations in parallel, which can make training **10–50x faster** than a CPU. Google Colab gives you a free GPU — make sure it's enabled: `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.


In [ ]:
# Core PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim

# torchvision gives us datasets and image transforms
import torchvision
import torchvision.transforms as transforms

# For loading data in batches and splitting datasets
from torch.utils.data import DataLoader, random_split

# For plotting and math
import matplotlib.pyplot as plt
import numpy as np

# For evaluation metrics
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# For reproducibility (so results are consistent across runs)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Print basic environment info
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

# Automatically use the GPU if Colab has given us one, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime -> Change runtime type -> GPU for faster training.")


## Section 2 — Load the CIFAR-10 Dataset (from local files)

**CIFAR-10** is a classic image classification dataset created by the Canadian Institute for Advanced Research. It contains:

- **60,000** color images total
- Split into **50,000 training images** (5 files: `data_batch_1` … `data_batch_5`, 10,000 images each) and **10,000 test images** (`test_batch`)
- **10 classes**: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck — their names live in `batches.meta`
- Each image is **32 x 32 pixels**, with **3 color channels** (Red, Green, Blue — RGB)

### Files to Upload

Upload these **7 files** (no file extensions) into a Colab folder — e.g. `/content/cifar-10-batches-py/`:

```
data_batch_1
data_batch_2
data_batch_3
data_batch_4
data_batch_5
test_batch
batches.meta
```

`readme.html` is **not** needed (it's just a page that redirects to the CIFAR-10 website).

**How to upload — pick ONE:**

**Option A (quick, per-session):** Run the upload cell below, click *Choose Files*, and select all 7 files at once. They'll be copied into `/content/cifar-10-batches-py/`. You'll need to re-upload every time your Colab runtime restarts.

**Option B (recommended if you'll reuse this often):** Upload the `cifar-10-batches-py` folder to your Google Drive once, then mount Drive in Colab and point `CIFAR10_ROOT` at that Drive path instead — no re-uploading needed on future sessions. A commented-out example is included below.

Each raw batch file is a **Python pickle** containing a dictionary with:
- `b'data'` — a NumPy array of shape `(10000, 3072)`, where each row is a flattened 32×32×3 image (3072 = 3 × 32 × 32)
- `b'labels'` — a list of 10,000 integer class labels (0–9)

We write a small custom `Dataset` class below to unpickle these files and reshape each row back into a proper `(32, 32, 3)` image.


In [ ]:
# --- Option A: upload the files directly into this Colab session ---
import os
CIFAR10_ROOT = "/content/cifar-10-batches-py"
os.makedirs(CIFAR10_ROOT, exist_ok=True)

from google.colab import files
print("Select all 7 files: data_batch_1..5, test_batch, batches.meta (skip readme.html)")
uploaded = files.upload()

import shutil
for fname in uploaded.keys():
    if fname == "readme.html":
        continue  # not needed
    shutil.move(fname, os.path.join(CIFAR10_ROOT, fname))

print("Files now in", CIFAR10_ROOT, ":", sorted(os.listdir(CIFAR10_ROOT)))

# --- Option B: use Google Drive instead (uncomment if you prefer this) ---
# from google.colab import drive
# drive.mount("/content/drive")
# CIFAR10_ROOT = "/content/drive/MyDrive/cifar-10-batches-py"  # adjust to your actual Drive path


In [ ]:
import pickle
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

def unpickle(file_path):
    """Load a single CIFAR-10 pickle file (returns a dict with byte-string keys)."""
    with open(file_path, "rb") as f:
        d = pickle.load(f, encoding="bytes")
    return d

class CIFAR10Local(Dataset):
    """Custom Dataset that reads the official CIFAR-10 'Python version' pickle files
    directly from disk, instead of using torchvision's auto-downloader."""

    def __init__(self, root, train=True, transform=None):
        self.transform = transform
        batch_files = [f"data_batch_{i}" for i in range(1, 6)] if train else ["test_batch"]

        data_list, label_list = [], []
        for bf in batch_files:
            batch = unpickle(os.path.join(root, bf))
            data_list.append(batch[b"data"])          # shape (10000, 3072)
            label_list.extend(batch[b"labels"])        # list of 10000 ints

        # Stack all batches -> (N, 3072) -> reshape to (N, 3, 32, 32) -> (N, 32, 32, 3) for PIL
        all_data = np.vstack(data_list)
        self.data = all_data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
        self.labels = label_list

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.fromarray(self.data[idx])  # NumPy (32,32,3) uint8 -> PIL Image
        label = self.labels[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, label

# Load class names from batches.meta
meta = unpickle(os.path.join(CIFAR10_ROOT, "batches.meta"))
classes = [name.decode("utf-8") for name in meta[b"label_names"]]

print("Number of classes:", len(classes))
print("Classes:", classes)

# Build the raw datasets (ToTensor only for now; full transforms come in Section 4)
raw_train_dataset = CIFAR10Local(CIFAR10_ROOT, train=True, transform=transforms.ToTensor())
raw_test_dataset = CIFAR10Local(CIFAR10_ROOT, train=False, transform=transforms.ToTensor())

print("Number of training images (before train/val split):", len(raw_train_dataset))
print("Number of test images:", len(raw_test_dataset))

# Look at one raw sample to confirm image size and channels
sample_image, sample_label = raw_train_dataset[0]
print("Single image tensor shape [C, H, W]:", sample_image.shape)
print("Example label:", sample_label, "->", classes[sample_label])


## Section 3 — Visualize the Dataset

Every CIFAR-10 image is stored as a tensor with shape **`[C, H, W]`**:

- **C = 3** — the number of color **channels** (Red, Green, Blue)
- **H = 32** — the **height** of the image in pixels
- **W = 32** — the **width** of the image in pixels

Each pixel value is a number between 0 and 1 (since we used `ToTensor()`, which also automatically scales raw 0–255 pixel values down to the 0–1 range).

Below, we display a 4x4 grid of 16 random training images along with their true class name as the title, so you can see what the data actually looks like.


In [ ]:
def imshow_grid(dataset, n=16, cols=4):
    rows = n // cols
    fig, axes = plt.subplots(rows, cols, figsize=(8, 8))
    indices = np.random.choice(len(dataset), n, replace=False)
    for ax, idx in zip(axes.flatten(), indices):
        image, label = dataset[idx]
        # image is [C, H, W] -> convert to [H, W, C] for matplotlib
        img_np = image.permute(1, 2, 0).numpy()
        ax.imshow(img_np)
        ax.set_title(classes[label], fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

imshow_grid(raw_train_dataset, n=16, cols=4)

# Print the tensor shape of a single image again, explained
img, lbl = raw_train_dataset[0]
print("Image tensor shape [C, H, W]:", img.shape)
print("  C (channels) =", img.shape[0], "-> Red, Green, Blue")
print("  H (height)   =", img.shape[1], "pixels")
print("  W (width)    =", img.shape[2], "pixels")


## Section 4 — Image Transforms

**Transforms** are preprocessing steps applied to every image before it's fed into the network.

For the **training set**, we add simple **data augmentation** — small random changes that make the model see slightly different versions of each image every epoch, which helps it generalize better and reduces overfitting:

- `RandomCrop(32, padding=4)` — randomly crops a 32x32 patch from a slightly padded image, shifting the object a little.
- `RandomHorizontalFlip()` — randomly flips the image left-right (a cat facing left is still a cat facing right).
- `ToTensor()` — converts the PIL image to a PyTorch tensor and scales pixels to [0, 1].
- `Normalize(mean, std)` — rescales each channel to roughly zero mean / unit variance, which helps training converge faster and more stably.

For the **validation and test sets**, we do **not** use augmentation — we only apply `ToTensor()` and `Normalize()`. This is because validation/test data should reflect the *real* images the model will see in deployment, not artificially altered ones. Augmenting test data would give a misleading measure of real-world performance.


In [ ]:
# Standard CIFAR-10 per-channel mean/std (commonly used values)
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),   # small random crop -> augmentation
    transforms.RandomHorizontalFlip(),      # random left-right flip -> augmentation
    transforms.ToTensor(),                  # convert image to tensor, scale to [0,1]
    transforms.Normalize(cifar_mean, cifar_std),  # normalize each channel
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

print("Training transform (with augmentation):")
print(train_transform)
print()
print("Test/Validation transform (no augmentation):")
print(test_transform)


## Section 5 — Train / Validation Split

We reload the training data twice — once with the augmentation transform and once with the plain (non-augmented) transform — because after splitting, the validation subset needs the *test-style* transform, not the augmented one.

We split the original **50,000** training images into:

- **80% (40,000 images) — Training set:** used to actually update the model's weights.
- **20% (10,000 images) — Validation set:** used only to *monitor* performance during training (never used to update weights). It tells us how well the model generalizes to unseen data while we're still developing it.

The official CIFAR-10 **test set (10,000 images)** is kept completely separate and untouched until Section 17, where it's used for the final, one-time evaluation.


In [ ]:
# Reload full training data with each transform style (reading the same local pickle files again)
full_train_aug = CIFAR10Local(CIFAR10_ROOT, train=True, transform=train_transform)
full_train_plain = CIFAR10Local(CIFAR10_ROOT, train=True, transform=test_transform)

# Test set uses the plain (no augmentation) transform
test_dataset = CIFAR10Local(CIFAR10_ROOT, train=False, transform=test_transform)

# Decide the split sizes: 80% train / 20% validation
total_size = len(full_train_aug)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

# Use a fixed generator so the split is reproducible
generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices = random_split(range(total_size), [train_size, val_size], generator=generator)

# Build Subset datasets: training subset gets augmentation, validation subset does not
from torch.utils.data import Subset
train_dataset = Subset(full_train_aug, train_indices.indices)
val_dataset = Subset(full_train_plain, val_indices.indices)

print("Training set size:  ", len(train_dataset))
print("Validation set size:", len(val_dataset))
print("Test set size:      ", len(test_dataset))


## Section 6 — DataLoaders

A **DataLoader** feeds the dataset to the model in small groups called **batches**, instead of all at once.

- **Batch size = 64** means the model looks at 64 images together, computes an average loss over them, and updates its weights once per batch — rather than updating after every single image (too slow and noisy) or after all 50,000 images at once (too memory-hungry and updates too rarely).
- **`shuffle=True`** for training means the order of images is randomized every epoch, which prevents the model from learning any accidental ordering in the data.
- **One epoch** = one full pass through the entire training dataset, i.e. going through *all* the batches once.

Number of batches per epoch = `ceil(dataset size / batch size)`.


In [ ]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("Batch size:", batch_size)
print("Number of training batches:  ", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:      ", len(test_loader))


## Section 7 — Understand the Batch Tensor Shape

Let's grab a single batch from `train_loader` and inspect its shape. You should see something like `[64, 3, 32, 32]`:

- **64** = batch size (number of images in this batch)
- **3** = RGB channels
- **32** = height in pixels
- **32** = width in pixels

The `labels` tensor will have shape `[64]` — one integer class label (0–9) per image in the batch.


In [ ]:
# Grab one batch
images, labels = next(iter(train_loader))

print("images.shape:", images.shape)
print("labels.shape:", labels.shape)
print()
print("Meaning of images.shape = [batch_size, channels, height, width]:")
print(f"  batch_size = {images.shape[0]} images in this batch")
print(f"  channels   = {images.shape[1]} (RGB)")
print(f"  height     = {images.shape[2]} pixels")
print(f"  width      = {images.shape[3]} pixels")
print()
print("First 10 labels in this batch:", labels[:10].tolist())
print("Corresponding classes:        ", [classes[l] for l in labels[:10].tolist()])


In [ ]:
# Visualize this batch (un-normalize first so colors look correct)
def unnormalize(img_tensor):
    mean = torch.tensor(cifar_mean).view(3, 1, 1)
    std = torch.tensor(cifar_std).view(3, 1, 1)
    return img_tensor * std + mean

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    img = unnormalize(images[i]).permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(img)
    ax.set_title(classes[labels[i]], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Section 8 — Build a Simple CNN

Now we design the neural network itself: a **Convolutional Neural Network (CNN)**, which is the standard architecture for image tasks because it's good at detecting local visual patterns (edges, textures, shapes) and reusing that ability across the whole image.

Our architecture:

```
Input: 3 x 32 x 32
Conv2d(3 -> 32, kernel=3, padding=1)  -> ReLU -> MaxPool2d(2,2)   # 32x32 -> 16x16
Conv2d(32 -> 64, kernel=3, padding=1) -> ReLU -> MaxPool2d(2,2)   # 16x16 -> 8x8
Flatten
Linear(64*8*8 -> 128) -> ReLU
Linear(128 -> 10)                      # 10 output scores, one per class
```

## Section 9 — Explaining the Architecture

| Layer | Output Shape | Purpose |
|---|---|---|
| Input | `[B, 3, 32, 32]` | Raw RGB image |
| Conv2d(3→32) + ReLU | `[B, 32, 32, 32]` | Learns 32 simple visual feature detectors (edges, colors, textures) |
| MaxPool2d(2,2) | `[B, 32, 16, 16]` | Shrinks the image, keeps the strongest signal in each 2x2 region |
| Conv2d(32→64) + ReLU | `[B, 64, 16, 16]` | Learns 64 more complex features built from the first layer's features |
| MaxPool2d(2,2) | `[B, 64, 8, 8]` | Shrinks again |
| Flatten | `[B, 4096]` | Turns the 3D feature map into a 1D vector |
| Linear(4096→128) + ReLU | `[B, 128]` | Combines all spatial features into a compact representation |
| Linear(128→10) | `[B, 10]` | Final class scores — **10 neurons because CIFAR-10 has 10 classes**, one score per class |

**Layer explanations:**

- **Conv2d (Convolution layer):** Slides small learnable filters (e.g. 3x3) across the image to detect local patterns like edges or color blobs. Early layers detect simple patterns; later layers combine them into complex ones (e.g. "wheel", "fur").
- **ReLU (activation function):** `ReLU(x) = max(0, x)`. It introduces non-linearity — without it, stacking layers would be mathematically equivalent to just one linear layer, and the network couldn't learn complex patterns.
- **MaxPooling:** Reduces the spatial size (height/width) by keeping only the maximum value in each small region. This makes the network faster, uses less memory, and makes it slightly more tolerant to small shifts in the image.
- **Flatten:** Reshapes the 3D feature maps `[C, H, W]` into a single 1D vector so it can be fed into a Linear layer.
- **Linear (Fully Connected) layer:** Every input neuron connects to every output neuron with a learnable weight — used to combine features into a final decision.
- **Output layer (10 neurons):** Each of the 10 output numbers is a raw "score" (called a *logit*) for one class. The highest score is the model's predicted class.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        # First convolution block: 3 input channels (RGB) -> 32 feature maps
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16

        # Second convolution block: 32 -> 64 feature maps
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8

        # Flatten + fully connected layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)  # 10 output scores (one per class)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)  # raw scores (logits) - no softmax here, see Section 11
        return x

model = SimpleCNN(num_classes=10).to(device)
print(model)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")


## Section 10 — Connecting This CNN to What You've Learned

Every `Conv2d` and `Linear` layer above contains **learnable parameters**: a **weight** matrix and a **bias** vector. Here's how the core deep learning concepts map onto this model:

```
Neuron
  -> Weight (how strongly an input influences a neuron's output)
  -> Bias (a learnable offset added to the weighted sum)
  -> Activation function, e.g. ReLU (adds non-linearity)
  -> Forward propagation (input flows through all layers to produce a prediction)
  -> Loss (how wrong the prediction was, compared to the true label)
  -> Backpropagation (loss "flows backward" to compute how much each weight contributed to the error)
  -> Gradient (the direction and size of the error contribution for each weight)
  -> Optimizer (uses the gradient to decide how to change each weight)
  -> Updated weights (the model gets slightly better)
```

In plain language:

- **Weight:** A number that scales how important an input is. Learned during training.
- **Bias:** An extra learnable number added after the weighted sum, giving the layer more flexibility.
- **Activation:** A function (like ReLU) applied after the weighted sum, letting the network model non-linear, complex relationships.
- **Gradient:** A measure of "if I nudge this weight slightly, how much does the loss change, and in which direction?" Computed automatically by PyTorch's autograd during `loss.backward()`.

This whole cycle — forward pass, loss, backward pass, weight update — repeats for every batch, for every epoch, and is exactly what Sections 13–15 implement.


## Section 11 — Loss Function

We use **`nn.CrossEntropyLoss()`**, the standard loss function for multi-class classification.

- CIFAR-10 has **10 classes**, so this is a multi-class problem (not binary).
- The model's raw output is 10 numbers (logits) per image, e.g. `[2.1, -0.4, 0.8, 0.1, -1.2, 3.4, -0.9, 0.2, -0.5, 1.0]`. The **largest** value corresponds to the model's predicted class.
- `CrossEntropyLoss` internally applies a **softmax** (which converts raw scores into probabilities that sum to 1) and then compares that probability distribution to the true label. **This is why we do NOT manually apply softmax in the model's `forward()` method** — doing so would apply softmax twice and hurt training.
- Intuitively, cross-entropy loss is **low** when the model assigns a high probability to the correct class, and **high** when it confidently predicts the wrong class.


In [ ]:
criterion = nn.CrossEntropyLoss()
print(criterion)


## Section 12 — Optimizer

The **optimizer** decides *how* to update the model's weights using the gradients computed during backpropagation.

We use **Adam**, a popular optimizer that adapts the learning rate for each parameter individually, which usually makes it converge faster and more reliably than plain gradient descent — a great default choice for beginners.

- **Learning rate (`lr = 0.001`):** controls *how big* each weight update step is. Too high → training becomes unstable; too low → training is very slow.
- The core update rule (simplified, plain gradient descent form) is:

```
w_new = w_old - learning_rate * gradient
```

Adam builds on this idea but also keeps track of the recent history of gradients to smooth out and scale the updates automatically.

**How optimizer + backpropagation work together:**
1. `loss.backward()` computes the gradient of the loss with respect to every weight.
2. `optimizer.step()` uses those gradients to actually update the weights.
3. `optimizer.zero_grad()` clears old gradients before the next batch (otherwise gradients would accumulate across batches).


In [ ]:
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(optimizer)


## Section 13 — Training Function

For **every batch** during training, we perform these steps:

1. Move `images` and `labels` to the `device` (GPU or CPU).
2. `optimizer.zero_grad()` — clear gradients left over from the previous batch.
3. **Forward pass**: `outputs = model(images)` — the model produces predictions.
4. **Compute loss**: `loss = criterion(outputs, labels)` — how wrong were the predictions?
5. **Backward pass**: `loss.backward()` — PyTorch automatically computes the gradient of the loss with respect to every weight.
6. **Update weights**: `optimizer.step()` — the optimizer nudges every weight using its gradient.
7. Track the running loss and accuracy for reporting.

We also call `model.train()` before this loop, which tells PyTorch layers like Dropout/BatchNorm (not used here, but good practice) to behave in "training mode".


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()              # 1. clear old gradients
        outputs = model(images)            # 2. forward pass -> predictions
        loss = criterion(outputs, labels)  # 3. compute how wrong we were
        loss.backward()                    # 4. backpropagation -> compute gradients
        optimizer.step()                   # 5. update weights using gradients

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)   # class with the highest score
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


## Section 14 — Validation Function

During validation we must NOT update the model's weights — we're only *checking* how well it currently performs on data it wasn't trained on this batch.

- **`model.eval()`** switches the model to evaluation mode.
- **`torch.no_grad()`** tells PyTorch not to track gradients at all. This saves memory and computation, since we won't call `backward()` here — we're not learning, just measuring.


In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()  # set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # no gradients needed - we are not training here
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


## Section 15 — Train the Model

We now train for **10 epochs**. For every epoch we print the training and validation loss/accuracy, so you can watch the model improve (or spot problems like overfitting) as it goes.

This is the longest-running cell in the notebook — with a Colab GPU it should take a few minutes for 10 epochs.


In [ ]:
num_epochs = 10

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc*100:.2f}%")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Accuracy: {val_acc*100:.2f}%")
    print("-" * 50)


## Section 16 — Plot Training History

Two plots help us understand training behavior:

1. **Training Loss vs Validation Loss** — both should generally decrease. If validation loss starts *increasing* while training loss keeps decreasing, that's a classic sign of **overfitting**: the model is memorizing the training data instead of learning general patterns.
2. **Training Accuracy vs Validation Accuracy** — both should generally increase. A large, growing gap between them (training much higher than validation) also signals overfitting.


In [ ]:
epochs_range = range(1, num_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_range, train_losses, label="Train Loss", marker="o")
axes[0].plot(epochs_range, val_losses, label="Validation Loss", marker="o")
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in train_accuracies], label="Train Accuracy", marker="o")
axes[1].plot(epochs_range, [a*100 for a in val_accuracies], label="Validation Accuracy", marker="o")
axes[1].set_title("Training vs Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Section 17 — Final Testing

Now, and only now, we evaluate on the **official CIFAR-10 test set** — the 10,000 images that were never used for training or validation. This gives an honest estimate of how the model would perform on brand-new, real-world data.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Final Test Loss:     {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc*100:.2f}%")


## Section 18 — Class-wise Accuracy

Overall accuracy hides an important detail: a model rarely performs equally well on every class. Here we compute accuracy separately for each of the 10 classes to see which ones the model struggles with (commonly, visually similar animal classes like cat/dog are confused more than, say, airplane/truck).


In [ ]:
model.eval()
class_correct = np.zeros(10)
class_total = np.zeros(10)

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct_mask = (predicted == labels)
        for i in range(labels.size(0)):
            label = labels[i].item()
            class_total[label] += 1
            class_correct[label] += correct_mask[i].item()

print(f"{'Class':<12}{'Accuracy':>10}")
print("-" * 22)
for i in range(10):
    acc = 100 * class_correct[i] / class_total[i]
    print(f"{classes[i]:<12}{acc:>9.2f}%")


## Section 19 — Confusion Matrix

A **confusion matrix** is a table where row *i* / column *j* shows how many images of true class *i* were predicted as class *j*. The diagonal represents correct predictions; anything off the diagonal is a mistake.

For example, if the cell at row "cat", column "dog" has a large number, it means the model frequently mistakes cats for dogs — a very common confusion, since both are furry four-legged animals photographed at similar angles.


In [ ]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Confusion Matrix - CIFAR-10 Test Set")
plt.tight_layout()
plt.show()

print(classification_report(all_labels, all_preds, target_names=classes))


## Section 20 — Correct and Incorrect Predictions

Looking at actual example images the model got right (and wrong) is one of the most useful debugging tools in deep learning — it often reveals *why* a model is confused (blurry images, unusual angles, genuinely ambiguous photos, etc.), which a single accuracy number can't tell you.


In [ ]:
def show_prediction_examples(model, dataset, device, num_correct=8, num_incorrect=8):
    model.eval()
    correct_examples = []
    incorrect_examples = []
    loader = DataLoader(dataset, batch_size=1, shuffle=True)

    with torch.no_grad():
        for images, labels in loader:
            if len(correct_examples) >= num_correct and len(incorrect_examples) >= num_incorrect:
                break
            images_dev = images.to(device)
            outputs = model(images_dev)
            _, predicted = torch.max(outputs, 1)
            pred = predicted.item()
            true = labels.item()

            if pred == true and len(correct_examples) < num_correct:
                correct_examples.append((images[0], true, pred))
            elif pred != true and len(incorrect_examples) < num_incorrect:
                incorrect_examples.append((images[0], true, pred))

    def plot_examples(examples, title):
        fig, axes = plt.subplots(1, len(examples), figsize=(2*len(examples), 2.5))
        fig.suptitle(title)
        for ax, (img, true, pred) in zip(axes, examples):
            img_show = unnormalize(img).permute(1, 2, 0).numpy().clip(0, 1)
            ax.imshow(img_show)
            ax.set_title(f"A:{classes[true]}\nP:{classes[pred]}", fontsize=8)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    plot_examples(correct_examples, "Correct Predictions")
    plot_examples(incorrect_examples, "Incorrect Predictions")

show_prediction_examples(model, test_dataset, device)


## Section 21 — Make a Prediction on One Image

The full prediction process for a single image is:

```
Image -> CNN -> 10 output scores -> Highest score -> Predicted class
```


In [ ]:
model.eval()
idx = np.random.randint(len(test_dataset))
image, true_label = test_dataset[idx]

with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))  # add batch dimension: [3,32,32] -> [1,3,32,32]
    scores = output.cpu().numpy()[0]
    predicted_label = int(np.argmax(scores))

plt.imshow(unnormalize(image).permute(1, 2, 0).numpy().clip(0, 1))
plt.title(f"Actual: {classes[true_label]} | Predicted: {classes[predicted_label]}")
plt.axis("off")
plt.show()

print("Raw output scores (one per class):")
for cls_name, score in zip(classes, scores):
    print(f"  {cls_name:<12}: {score:.2f}")
print(f"\nHighest score -> predicted class: {classes[predicted_label]}")


## Section 22 — Save the Model

A **`.pth`** file stores the model's **`state_dict`** — a Python dictionary mapping each layer's name to its learned weight and bias tensors. It does *not* store the model's code/architecture, so to reload it, you must first recreate the same `SimpleCNN` class and then load the saved weights into it.


In [ ]:
# Save just the learned weights (recommended approach)
torch.save(model.state_dict(), "cifar10_cnn.pth")
print("Model weights saved to cifar10_cnn.pth")

# Demonstrate loading it back into a fresh model instance
loaded_model = SimpleCNN(num_classes=10).to(device)
loaded_model.load_state_dict(torch.load("cifar10_cnn.pth", map_location=device))
loaded_model.eval()
print("Model successfully reloaded from cifar10_cnn.pth")

# Quick sanity check: loaded model should give the same test accuracy
_, reloaded_acc = evaluate(loaded_model, test_loader, criterion, device)
print(f"Reloaded model test accuracy: {reloaded_acc*100:.2f}% (should match Section 17's result)")


## Section 23 — Important Concept Summary

| Concept | Meaning |
|---|---|
| Neuron | A single computational unit that takes weighted inputs, adds a bias, and applies an activation function |
| Weight | A learnable number controlling how strongly an input affects a neuron's output |
| Bias | A learnable offset added to a neuron's weighted sum |
| Activation | A function (e.g. ReLU) applied after the weighted sum, adding non-linearity |
| CNN | Convolutional Neural Network — a network specialized for grid-like data such as images |
| Convolution | Sliding small learnable filters across an image to detect local patterns |
| Pooling | Reducing spatial size by summarizing regions (e.g. taking the max value) |
| Forward Propagation | Passing input through the network to produce a prediction |
| Loss | A number measuring how wrong the model's prediction was |
| Backpropagation | Algorithm that computes how much each weight contributed to the loss |
| Gradient | The rate of change of the loss with respect to a weight |
| Optimizer | Algorithm (e.g. Adam) that uses gradients to update weights |
| Learning Rate | How big each weight update step is |
| Batch Size | Number of samples processed together before one weight update |
| Epoch | One full pass through the entire training dataset |
| Training | The process of updating weights to reduce loss on the training set |
| Validation | Checking performance on unseen data during development, without updating weights |
| Testing | Final, one-time evaluation on completely untouched data |
| Overfitting | When a model performs well on training data but poorly on new/unseen data |


## Section 24 — Complete Training Pipeline

```
Dataset
  -> Preprocessing (transforms: augmentation, normalization)
  -> DataLoader (splits data into batches)
  -> Batch
  -> CNN
  -> Forward Pass (produces predictions)
  -> Prediction (10 class scores)
  -> Loss (compares prediction to true label)
  -> Backpropagation (computes gradients)
  -> Gradient
  -> Adam Optimizer (updates weights using gradients)
  -> Update Weights
  -> Next Batch  (repeat until all batches in this epoch are done)
  -> Next Epoch  (repeat the whole process for more epochs)
  -> Validation  (check generalization performance after each epoch, no weight updates)
  -> Final Test  (one-time evaluation on the untouched test set, after all training is done)
```

In plain language: the raw images are cleaned and organized into small batches, fed through the CNN to get predictions, compared against the true labels to get a loss, and that loss is used (via backpropagation and the optimizer) to nudge the weights slightly better. This repeats for many batches and many epochs, while validation keeps an eye on whether the model is actually learning to generalize — and only at the very end do we check the final, honest test accuracy.


## Section 25 — Practical Experiments (Do These Yourself)

Try each of the following by changing the relevant cell above and re-running. For each experiment, record: **what you changed, why, what you observed, and your conclusion** — this is exactly what goes into your lab report (Section 27).

### Experiment 1 — Learning Rate
Change `learning_rate` in Section 12 to `0.01`, then `0.001` (default), then `0.0001`.
- **What to observe:** Does the loss decrease smoothly, oscillate wildly, or barely move at all?
- **Expect:** Too high (0.01) may cause unstable/oscillating loss; too low (0.0001) will train very slowly.

### Experiment 2 — Batch Size
Change `batch_size` in Section 6 to `32`, then `64` (default), then `128`.
- **What to observe:** Training speed (time per epoch) and final accuracy.
- **Expect:** Larger batches are usually faster per epoch (more parallelism) but may need more epochs or tuning to reach the same accuracy.

### Experiment 3 — Number of Epochs
Change `num_epochs` in Section 15 to `5`, then `10` (default), then `20`.
- **What to observe:** Watch the Section 16 plots — does the gap between train and validation curves widen at higher epoch counts?
- **Expect:** With more epochs, training accuracy keeps climbing but validation accuracy may plateau or even drop — a sign of overfitting.

### Experiment 4 — Remove a Convolution Layer
In `SimpleCNN` (Section 8), remove the second conv block (`conv2`, `relu2`, `pool2`) and adjust `fc1`'s input size accordingly (it will now be `32 * 16 * 16`).
- **What to observe:** Compare test accuracy and parameter count to the original two-conv-layer model.
- **Expect:** Usually lower accuracy, since the network has less capacity to learn complex features — though it will also be faster to train.

### Experiment 5 — Change Optimizer
Replace `torch.optim.Adam(...)` in Section 12 with `torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)`.
- **What to observe:** Does training converge faster/slower than Adam? Is the final accuracy similar?
- **Expect:** SGD often needs more epochs and careful learning-rate tuning to match Adam's convergence speed, but can sometimes generalize slightly better with enough training.


## Section 26 — Viva / Interview Questions with Answers

1. **What is image classification?** Assigning a single category label to an entire input image.
2. **What is CIFAR-10?** A benchmark dataset of 60,000 32x32 color images across 10 classes (50,000 train / 10,000 test).
3. **Why does CIFAR-10 have 10 output neurons?** Because there are exactly 10 possible classes, and the model outputs one score per class.
4. **What is a CNN?** A neural network that uses convolutional layers to detect spatial patterns in grid-like data such as images.
5. **What is convolution?** Sliding a small learnable filter across an image and computing dot products to produce a feature map.
6. **What is pooling?** Downsampling a feature map by summarizing regions (e.g. taking the maximum value), reducing size and computation.
7. **What is ReLU?** An activation function, `max(0, x)`, that adds non-linearity so the network can model complex functions.
8. **What is a batch?** A small group of samples processed together in one forward/backward pass before a single weight update.
9. **What is an epoch?** One complete pass through the entire training dataset.
10. **What is a loss function?** A function that measures how far the model's predictions are from the true labels.
11. **Why use CrossEntropyLoss?** It's the standard loss for multi-class classification; it combines softmax and negative log-likelihood in one numerically stable step.
12. **What is backpropagation?** The algorithm that computes the gradient of the loss with respect to every weight by propagating the error backward through the network.
13. **What is a gradient?** The rate of change of the loss with respect to a given weight — it tells the optimizer which direction to move that weight.
14. **What is an optimizer?** An algorithm that uses gradients to update the model's weights to reduce the loss (e.g. Adam, SGD).
15. **What is Adam?** An adaptive optimizer that adjusts the effective learning rate per parameter using running estimates of past gradients, usually converging faster than plain SGD.
16. **What is learning rate?** A hyperparameter controlling how large each weight update step is.
17. **Why do we use validation data?** To monitor how well the model generalizes to unseen data during development, without touching the test set.
18. **What is overfitting?** When a model learns the training data too specifically (including its noise), performing well on training data but poorly on new data.
19. **Why do we use `model.train()`?** It puts the model in training mode, which affects layers like Dropout/BatchNorm that behave differently during training vs. inference.
20. **Why do we use `model.eval()`?** It puts the model in evaluation mode, ensuring consistent, deterministic behavior during validation/testing (and is paired with `torch.no_grad()` for efficiency).


## Section 27 — Lab Report Template

Copy this template into your lab report and fill it in using the **actual results you got when you ran this notebook** — do not invent numbers.

---

**1. Experiment Title**
Image Classification Using CNN on CIFAR-10 with PyTorch

**2. Objective**
To design, train, and evaluate a Convolutional Neural Network for classifying CIFAR-10 images into 10 categories, and to understand the core deep learning training pipeline.

**3. Dataset Description**
CIFAR-10: 60,000 32x32 RGB images, 10 classes, split as 50,000 train (further split 80/20 into train/validation) and 10,000 test images.

**4. Tools and Libraries**
Python, PyTorch, torchvision, matplotlib, numpy, scikit-learn, seaborn, Google Colab (GPU runtime).

**5. Model Architecture**
_(Paste the printed `model` summary and total parameter count from Section 9.)_

**6. Training Configuration**
- Batch size: `___` (fill in from Section 6)
- Learning rate: `___` (fill in from Section 12)
- Optimizer: `___`
- Loss function: CrossEntropyLoss
- Number of epochs: `___`

**7. Training Results**
_(Fill in the final epoch's train loss and train accuracy from Section 15's printed output.)_

**8. Validation Results**
_(Fill in the final epoch's validation loss and validation accuracy from Section 15's printed output.)_

**9. Test Results**
_(Fill in the test loss and test accuracy printed in Section 17.)_

**10. Confusion Matrix**
_(Paste or describe the heatmap generated in Section 19, and note which class pairs were most confused.)_

**11. Observations**
_(Describe trends from the loss/accuracy plots in Section 16 — did loss decrease steadily? did any overfitting appear? which classes had the lowest per-class accuracy in Section 18?)_

**12. Problems / Limitations**
_(e.g. small/simple architecture, limited epochs, CIFAR-10's low 32x32 resolution making some classes hard to distinguish, etc.)_

**13. Conclusion**
_(Summarize what was learned and how the model could be improved — e.g. deeper network, more epochs, learning rate scheduling, stronger augmentation, etc.)_


## Optional Appendix — Using `torchvision`'s Auto-Downloader Instead

The main notebook above loads CIFAR-10 from the **local pickle files you uploaded** in Section 2 (`CIFAR10Local`). If you ever want to skip manual file uploads entirely and let PyTorch fetch the dataset automatically instead (e.g. on a different machine where you don't have the files handy), you can replace Section 2's loading code with this simpler alternative:


In [ ]:
# --- OPTIONAL ALTERNATIVE: auto-download CIFAR-10 via torchvision instead of local files ---
# raw_train_dataset = torchvision.datasets.CIFAR10(
#     root="./data", train=True, download=True, transform=transforms.ToTensor()
# )
# raw_test_dataset = torchvision.datasets.CIFAR10(
#     root="./data", train=False, download=True, transform=transforms.ToTensor()
# )
# classes = raw_train_dataset.classes
#
# This downloads the same CIFAR-10 data directly from the University of Toronto's
# servers and caches it locally, so no manual file upload is needed. Everything else
# in this notebook (Sections 3 onward) works unchanged either way, since both
# CIFAR10Local and torchvision.datasets.CIFAR10 return (image, label) pairs.
print("This cell is optional and commented out by default. See the markdown above for details.")
